# Phyiosological features of neurons
In this chapter we are going to cover the main physiologial features of neurons. While the ion chapter covered how different channel are expressed this chapter is going to go over the features of neurons as a point of reference and what ion channels contribute to those differences. We will also cover some general features and why they occur physiologically. Some of this may seem to repeat what is in the [Current Clamp](#current_clamp) chapter. However, that chapter is concerned more is how we analyze these features rather than the physiological basis of these features.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from dotenv import find_dotenv
from scipy import signal

env_file = find_dotenv()

base_path = Path(env_file).parent / "data"

with open(base_path / "spiking_neurons/pv.json", "r") as rf:
    pv = np.array(json.load(rf)["array"])
with open(base_path / "spiking_neurons/pyr_adapting.json", "r") as rf:
    pyr_adapt = np.array(json.load(rf)["array"])
with open(base_path / "spiking_neurons/pyr_doublet.json", "r") as rf:
    pyr_doublet = np.array(json.load(rf)["array"])
with open(base_path / "spiking_neurons/msn.json", "r") as rf:
    msn = np.array(json.load(rf)["array"])

## Spike frequency adaptation
Spike frequency adaptation (SFA) is when the instaneous spike frequency decreases during a spike train. SFA is classically seen in cortical pyramidal neurons. If you compare pyramidal neuron, MSN and PV interneuron you can see the prominent slowing of the spike frequency. We can see the ISI increases in the pyramidal neuron but does not in the PV interneuron and MSN.

In [ ]:
def plot_isi_width(array, ax):
    peaks, _ = signal.find_peaks(array, height=10)
    top = np.max(array[peaks])
    for i in range(len(peaks) - 1):
        t = (peaks[i + 1] - peaks[i]) / 10
        ax.plot(
            [peaks[i], peaks[i + 1]],
            [top + (i + 1) % 2 * 2 + 1, top + (i + 1) % 2 * 2 + 1],
            color="black",
            alpha=0.8,
        )
        ax.annotate(
            t,
            xy=(
                peaks[i] + ((peaks[i + 1] - peaks[i]) / 2),
                top + (i + 1) % 2 * 2 + 1,
            ),  # Target location in data coordinates
            xycoords="data",
            xytext=(0.0, 0.0),  # Offset: 8 points left, 0 points vertical
            textcoords="offset points",
            ha="center",  # Align right edge of text to the offset point
            va="bottom",
            fontsize=10,
        )


fig, ax = plt.subplots(ncols=3, layout="constrained", sharey=True, figsize=(10, 4))
ax[0].plot(pv[6000:7300], color="magenta")
ax[0].set_title("PV")
plot_isi_width(pv[6000:7300], ax[0])
ax[1].plot(pyr_adapt[5000:15000], color="magenta")
ax[1].set_title("Pyramidal")
plot_isi_width(pyr_adapt[5000:15000], ax[1])
ax[2].plot(msn[4500:11000], color="magenta")
ax[2].set_title("MSN")
plot_isi_width(msn[4500:11000], ax[2])
lims = [i.get_ylim() for i in ax]
max_lim = max(lims, key=lambda x: x[0])[0]
min_lim = min(lims, key=lambda x: x[1])[1]
for i in ax:
    i.set_axis_off()
    i.set_ylim(max_lim, min_lim)

SFA is a slow timescale phenomena. The primary reason the SFA occurs is due to the inability of voltage-gated Na+ channels, like Nav1.1 (Scna1), to reset after a spike occurs. Voltaged-gated Na+ channels need a hyperpolarized voltage to reactivate. This hyperpolarization is driven by BK, SK and Ca++ currents (which a lot of K+ currents). Many voltage-gated ion channels have a population time constant for how long it takes for population to be able to open again. If you reduce K+, or the Ca++ currents that activate some K+ channels, you can reduce the size of the AHP but at the potential cost of reducing the number of voltage-gated Na+ channels that are reactivated. PV interneurons have a massive AHP, driven in part by a specific BK channel subunit. This helps reset most of the voltage-gated Na+ channels. Where as with MSNs and pyramidal neurons, the AHP is much smaller, this means that sodium channels will need more time to reactivate since the voltage did not drop low enough to fully reactivate them which limits the firing rate of the neuron. Lastly we need to Ca++ to fully explain SFA. Ca++ can build up in a cell during long spike trains causing prolonged SK channel activation and lengthing the mAHP in pyramidal neurons. We can see that if we convolve a spike train (from a real pyramidal cell with SFA) with an exponential function we can see how the theoretical calcium concentration builds up.

In [ ]:
peaks = np.array([5094, 5372, 5726, 6196, 6680, 7149, 7707, 8259, 8916, 9786, 10553])
x = np.zeros(peaks[-1] - peaks[0] + 1500)
x[peaks - peaks[0]] = 1

tau_decay = 2000
tau_rise = 100
t_kernel = np.arange(tau_decay * 4)

# Bi-exponential kernel (use single exponential by omitting the second term)
kernel = np.exp(-t_kernel / tau_decay) - np.exp(-t_kernel / tau_rise)
kernel /= np.max(kernel)  # Normalize peak response to 1

# --- 3. Convolution & Truncation ---
output = signal.convolve(x, kernel, mode="full")[: len(x)]
fig, ax = plt.subplots(layout="constrained")
ax.plot(np.arange(output.size)/10, output, color="black")
ax.set_xlabel("Time (ms)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.set_yticks([])
_ = ax.set_title("Simulated Ca++ Levels")

Some neurons express high levels of Ca++ binding proteins, like parvalbumin, so build up of Ca++ does not occur. Additionally, pyramidal cells have back propogating action potentials (bAPs) that causing calcium influx in the dendrites that in turn comes back to the soma. PV neurons and other neurons do not have bAPs due to a combination of small dendrites (strong filtering) and lack of specific ion channels in the dendrites[@huFastspikingParvalbuminGABAergic2014]. Lastly, there is something called the slow AHP (sAHP). The sAHP is thought to partly regulate the SFA seen in pyramidal neurons [@higgsDiversityGainModulation2006;@andradeCalciumactivatedSlowAHP2012]. The sAHP activates slowly and decays slowly (100s of milliseconds). The sAHP is in part driven by calcium may involve calcium sensors and potassium channels [@andradeCalciumactivatedSlowAHP2012]. So far the best way I have seen to see the AHP is to do a short high frequency burst of spikes, usually doing ten 3-5 ms suprathreshold (far above rheobase) current injections spaced about 15 ms apart (50-60 Hz)[@guanElectrophysiologicalPropertiesGenetically2015]. After the last spike you should see the sAHP in cells as a long hyperpolarized (below the resting membrane potential) membrane voltage.

The computational function of SFA is pretty interesting. If a pyramidal neuron is recieving oscillatory input as a single frequency it will not necessarily spike at that output. Whereas PV, MSNs and other cell types that do not adapt can output the frequency of input they are recieving (at least up to a their physiological limit). The mathematical term for the oscillatory decorrelation that we see in pyramidal neurons exhibiting SFA is called whitening [@pozzoriniTemporalWhiteningPowerlaw2013]. Whitening increases the computation capacity of the brain.

## Spike repolarization and the AHP

Spike repolarization and the AHP is a process that starts once the cell passes is spike threshold. If we pick a single spike from a PV interneuron, pyramidal neuron and a MSN, we can see several important differences in the spike and the AHP. First, we can see that the peak spike voltage is different between the cell types. Then the peak AHP (minimum) is largest in the PV neuron. Then We can see that the shape of the AHP is quite different between the cell types.

In [ ]:
def plot_repol(array, ax):
    ax.plot(array, color="magenta")
    val = round(array.min(), 2)
    val_index = array.argmin()
    ax.axhline(val, color="black", alpha=0.3, linestyle="dashed")
    ax.annotate(
        f"{val} mV",
        xy=(val_index, val),  # Target location in data coordinates
        xycoords="data",
        xytext=(0.0, -3),  # Offset: 8 points left, 0 points vertical
        textcoords="offset points",
        ha="center",  # Align right edge of text to the offset point
        va="top",
        fontsize=10,
    )
    val = round(array.max(), 2)
    val_index = array.argmax()
    ax.axhline(val, color="black", alpha=0.3, linestyle="dashed")
    ax.annotate(
        f"{val} mV",
        xy=(val_index, val),  # Target location in data coordinates
        xycoords="data",
        xytext=(0.0, 3),  # Offset: 8 points left, 0 points vertical
        textcoords="offset points",
        ha="left",  # Align right edge of text to the offset point
        va="bottom",
        fontsize=10,
    )


fig, ax = plt.subplots(ncols=3, layout="constrained", sharey=True, figsize=(4.5, 4))
plot_repol(pv[6100:6300], ax[0])
ax[0].set_title("PV")
plot_repol(pyr_adapt[6050:7400], ax[1])
ax[1].set_title("Pyramidal")
plot_repol(msn[4950:5800], ax[2])
ax[2].set_title("MSN")
for i in ax:
    i.set_axis_off()

### Spike repolarization
Spike repolarization is driven by BK channels; large current Ca++ and voltaged-gated potassium channels. Cortical interneurons express a BK channel subunit, KCNMB2, that increases the calcium and voltage sensitivity of the channels and decreases the time it spends open with an intracellular gating mechanism [@liModulationBKChannel2016; @contetBKChannelsCentral2016]. So the BK channels turn on sooner, truncating the the action potential and turn off quickly shortening the fAHP. However, Kv3 channels also play a role. Kv3 channels are only activated at higher voltages (close to 0 mV) and have fast activation and deactivation kinetics[@covarrubiasStructuralInsightsFunction2025]. They are highly expressed by PV interneurons and another important factor in allow PV neurons to have a short spike and deep AHP.

### AHP
While the peak AHP is driven by BK and some smaller conductance K+ channels, the primary drivers of the AHP shape are the SK channels and Ca++. SK channels are small conductance channels that are both Ca++ and voltage-gated. When a spike initiates is causing release of Ca++ through a variety of voltage-gated Ca++ channels. This Ca++ in turn can cause SK channels to open. Prolonged SK channel opening can lengthen the AHP. Another factor effecting the AHP can be HCN channels. These channels open when the voltage drops below -40 to -50 mV and can push the membrane voltage even lower. Even though HCN channels cause hyperpolarization they are well known to cause rebound firing. HCN channels are important for rhythmic firing in a lot of spontaneously firing cells.

## Spike doublets and burst firing in pyramidal neurons
Pyramidal neurons can firing short, high frequency bursts when stimulated with a large current injection. At lower current injections they also have doublet or triplet or short ISI spikes. These are due to back propogating action potentials (BAPs), denritic calcium and the dendritic structure itself. When a pyramidal neuron initiates an action potential at the soma/axon hillock, the Na+ wave can travel back into the dendrites causing T-type Ca++ channels to open which then travel back to the soma eliciting another spike earlier than expected. Additionally, if you inject a depolarizing current into the dendrite you induce a doublet even at low currents. The dendritic Ca++ currents can get large enough to elicit a spike indepedent of Na+. While cell types have some Ca++ increase in the dendrites not all cells have BAPs. Some neurons, like PV interneurons, have no somatic action potential induced dendritic Ca++ or BAPs[@huFastspikingParvalbuminGABAergic2014].

In [ ]:
fig, ax = plt.subplots()
ax.plot(pyr_doublet[3000:10000], color="magenta")
ax.annotate(
    "",
    xy=(1099, 40),
    xytext=(1099, 55),
    arrowprops={"arrowstyle": "simple", "facecolor": "black"},
)
_ = ax.set_ylim((None, 55 * 0.05 + 55))
ax.set_axis_off()